<a href="https://colab.research.google.com/github/raksd/LLM_finetuning/blob/main/GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y torchao
!pip install -q trl peft datasets accelerate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.5/925.5 kB 2.4 MB/s eta 0:00:00


In [ ]:
import re, torch
from datasets import load_dataset
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

SYSTEM = "Solve the math problem. Think step by step, then give the final number as <answer>NUMBER</answer>."

def prep(ex):
    return {
        "prompt": [{"role": "system", "content": SYSTEM},
                   {"role": "user", "content": ex["question"]}],
        "gold": ex["answer"].split("####")[-1].strip().replace(",", ""),
    }

raw = load_dataset("openai/gsm8k", "main")
train_ds = raw["train"].map(prep, remove_columns=["question", "answer"])
test_ds  = raw["test"].select(range(100)).map(prep, remove_columns=["question", "answer"])


print("Train size:", len(train_ds), "| Test size:", len(test_ds))
print("\n--- RAW example ---")
print("Question:", raw["train"][0]["question"])
print("Full solution:", raw["train"][0]["answer"])
print("\n--- AFTER prep() ---")
print("Prompt sent to model:", train_ds[0]["prompt"])
print("Gold answer kept:", train_ds[0]["gold"])



def extract(text):
    m = re.search(r"<answer>\s*(-?[\d,\.]+)\s*</answer>", text)
    return m.group(1).replace(",", "").rstrip(".") if m else None

def correctness_reward(completions, gold, **kw):
    return [2.0 if extract(c[0]["content"]) == g else 0.0 for c, g in zip(completions, gold)]

def format_reward(completions, **kw):
    return [0.5 if extract(c[0]["content"]) is not None else 0.0 for c in completions]


# Fake model answers so you can see the scoring
fake = [
    [{"role": "assistant", "content": "5+3=8. <answer>8</answer>"}],    # right + formatted
    [{"role": "assistant", "content": "5+3=9. <answer>9</answer>"}],    # wrong + formatted
    [{"role": "assistant", "content": "The answer is 8."}],              # right but no tags
]
gold = ["8", "8", "8"]
print("correctness:", correctness_reward(fake, gold))   # expect [2.0, 0.0, 0.0]
print("format:     ", format_reward(fake))              # expect [0.5, 0.5, 0.0]




README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
tok.padding_side = "left"

def accuracy(model, ds):
    model.eval(); correct = 0
    for ex in ds:
        ids = tok.apply_chat_template(ex["prompt"], add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**ids, max_new_tokens=256, do_sample=False)
        text = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        correct += extract(text) == ex["gold"]
    return correct / len(ds)

base = AutoModelForCausalLM.from_pretrained(name, device_map={"": 0}, torch_dtype=torch.float16)
print("Base accuracy:", accuracy(base, test_ds))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base accuracy: 0.04


In [ ]:
results = {"base": accuracy(base, test_ds)}
print(results)

{'base': 0.04}


In [ ]:
import gc
del base; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from trl import SFTTrainer, SFTConfig

def to_messages(ex):
    reasoning, gold = ex["answer"].split("####")
    reasoning = re.sub(r"<<.*?>>", "", reasoning).strip()   # remove calculator annotations
    gold = gold.strip().replace(",", "")
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": f"{reasoning}\n<answer>{gold}</answer>"},
    ]}

sft_ds = raw["train"].select(range(2000)).map(to_messages, remove_columns=raw["train"].column_names)

sft_trainer = SFTTrainer(
    model=name,
    train_dataset=sft_ds,
    args=SFTConfig(
        output_dir="./sft-qwen", num_train_epochs=1,
        per_device_train_batch_size=4, gradient_accumulation_steps=2,
        learning_rate=2e-4, logging_steps=20, save_strategy="no",
        fp16=True, report_to="none",
    ),
    peft_config=LoraConfig(r=16, lora_alpha=32,
                           target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                           task_type="CAUSAL_LM"),
)
sft_trainer.train()
results["sft_lora"] = accuracy(sft_trainer.model, test_ds)
print(results)

del sft_trainer; gc.collect(); torch.cuda.empty_cache()

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Step,Training Loss
20,0.841655
40,0.402801
60,0.364895
80,0.356678
100,0.352221
120,0.364887
140,0.369203
160,0.353393
180,0.365628
200,0.342007


{'base': 0.04, 'sft_lora': 0.3}


In [ ]:
results["grpo"] = accuracy(trainer.model, test_ds)
print(results)

NameError: name 'trainer' is not defined

In [ ]:
import pandas as pd, json
df = pd.DataFrame({"model": results.keys(), "accuracy_%": [round(v * 100, 1) for v in results.values()]})
print(df.to_string(index=False))
json.dump(results, open("results.json", "w"))